# 2.1 总体、样本与经验分布

## 2.1.1 总体（Population）

统计推断的起点，是承认一个现实：**我们永远无法观测到所有数据。**

以A股市场为例。假设我们关心的是沪深300指数的日收益率。从理论上说，"总体"（population）是指**所有可能发生的交易日收益率**——包括过去已经发生的、未来将要发生的，乃至在同样的市场条件下本可能发生但未曾发生的。这个集合是无限的，也是永远无法完整观测的。

我们用一个概率分布来刻画总体：

$$Y \sim F(\cdot\,;\,\theta)$$

其中 $Y$ 是随机变量（这里是日收益率），$F$ 是其累积分布函数（CDF），$\theta$ 是刻画该分布的参数。以正态分布为例，$\theta = (\mu, \sigma^2)$，分别对应均值（预期收益率）和方差（波动率的平方）。

**$\theta$ 是我们想知道但永远无法精确知道的量。** 统计推断的全部工作，就是从有限的样本出发，对 $\theta$ 作出尽可能可靠的估计与判断。

> **金融视角**：在资产定价中，$\mu$ 对应资产的预期收益率，$\sigma$ 对应风险（波动率）。Markowitz 均值-方差框架的全部输入，都是对总体参数 $\theta$ 的估计——而这些估计本身带有误差，这正是"估计误差"（estimation error）问题的根源。

---

## 2.1.2 样本（Sample）

由于总体不可完全观测，我们只能依赖**样本**（sample）——从总体中抽取的 $n$ 个观测值：

$$\{y_1, y_2, \ldots, y_n\}$$

**独立同分布（IID）假设**是经典统计推断的基石：每个 $y_i$ 独立地来自同一个分布 $F(\cdot;\theta)$。然而，金融数据天然地对这一假设构成挑战：

- **非独立**：今天的波动率往往与昨天的波动率相关（波动率聚集，ARCH/GARCH效应）；
- **非同分布**：市场结构会随时间演化，2008 年的收益率分布与 2019 年的并不相同。

因此，在金融实证中，使用历史样本时需要在**样本量**（越长的历史，$n$ 越大，估计越精确）与**平稳性**（越短的窗口，越接近当前的市场状态）之间作出取舍。这个张力贯穿金融计量的始终。

**统计量（Statistic）** 是样本的函数 $T = g(y_1, \ldots, y_n)$，是对原始样本信息的压缩。常见的统计量包括：

$$\bar{y} = \frac{1}{n}\sum_{i=1}^n y_i, \quad s^2 = \frac{1}{n-1}\sum_{i=1}^n (y_i - \bar{y})^2$$

统计量保留了样本的部分信息，但也不可避免地丢弃了另一部分。**选择合适的统计量**，是统计推断的核心技艺之一（这将在第X章讨论充分统计量与MLE时深入展开）。

---

## 2.1.3 经验分布（Empirical Distribution）

拥有样本 $\{y_1, \ldots, y_n\}$ 之后，最自然的问题是：**我们能否直接从数据本身估计总体分布 $F$，而不预设任何参数形式？**

**经验累积分布函数（ECDF）** 给出了一个直接的回答：

$$\hat{F}_n(x) = \frac{1}{n}\sum_{i=1}^n \mathbf{1}(y_i \leq x)$$

直觉上，$\hat{F}_n(x)$ 就是样本中**不超过 $x$ 的观测值所占的比例**。它是一个阶梯函数，每个观测值处跳跃 $1/n$。

**Glivenko-Cantelli 定理**（"统计学基本定理"）保证了 ECDF 的一致收敛性：

$$\sup_x \left|\hat{F}_n(x) - F(x)\right| \xrightarrow{a.s.} 0 \quad \text{as } n \to \infty$$

即随着样本量增大，经验分布以概率1一致收敛于真实分布。这是**非参数推断**（如 Bootstrap）的理论基础：当 $n$ 足够大时，我们可以用经验分布代替真实分布进行推断。





**📓 Notebook 示例 2.1：沪深300收益率的经验分布**

下面，我们用真实数据来看看：沪深300的日收益率，究竟与正态分布有多大差距？

```python
# ============================================================
# Notebook 示例 2.1：总体、样本与经验分布
# 数据：沪深300指数 (000300.SS) 日收益率
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.distributions.empirical_distribution import ECDF

# ── 1. 获取数据 ──────────────────────────────────────────────
# 使用 akshare 获取沪深300真实数据（或读取本地CSV）
# pip install akshare
import akshare as ak

df_raw = ak.index_zh_a_hist(
    symbol="000300", period="daily",
    start_date="20050101", end_date="20231231"
)
df_raw.index = pd.to_datetime(df_raw["日期"])
price = df_raw["收盘"]

# 计算日对数收益率
ret = np.log(price / price.shift(1)).dropna()
ret.name = "CSI300_LogReturn"

print(f"样本量 n = {len(ret)}")
print(f"样本区间：{ret.index[0].date()} ~ {ret.index[-1].date()}")
print(f"\n描述统计：")
print(ret.describe())
print(f"偏度（Skewness）: {ret.skew():.4f}")
print(f"超额峰度（Excess Kurtosis）: {ret.kurtosis():.4f}")
```

```python
# ── 2. 绘制 ECDF vs 正态分布 CDF ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 拟合正态分布参数（MLE = 样本均值、样本标准差）
mu_hat, sigma_hat = ret.mean(), ret.std()
x_grid = np.linspace(ret.quantile(0.001), ret.quantile(0.999), 500)

# 左图：ECDF vs 正态 CDF
ecdf = ECDF(ret.values)
axes[0].step(ecdf.x, ecdf.y, color='steelblue', lw=1.5,
             label='ECDF（实际数据）')
axes[0].plot(x_grid, stats.norm.cdf(x_grid, mu_hat, sigma_hat),
             color='tomato', lw=2, linestyle='--',
             label=f'正态 CDF（μ={mu_hat:.4f}, σ={sigma_hat:.4f}）')
axes[0].set_title('经验分布 vs 正态分布')
axes[0].set_xlabel('日对数收益率')
axes[0].set_ylabel('累积概率')
axes[0].legend()
axes[0].grid(alpha=0.3)

# 右图：直方图 + 正态密度 + 核密度估计（KDE）
axes[1].hist(ret, bins=100, density=True, color='steelblue',
             alpha=0.4, label='收益率直方图')
axes[1].plot(x_grid, stats.norm.pdf(x_grid, mu_hat, sigma_hat),
             color='tomato', lw=2, linestyle='--', label='正态 PDF')

# 核密度估计（非参数，不假定任何分布形式）
from scipy.stats import gaussian_kde
kde = gaussian_kde(ret)
axes[1].plot(x_grid, kde(x_grid), color='navy', lw=2, label='核密度估计（KDE）')

axes[1].set_title('收益率分布：正态假设 vs 实际')
axes[1].set_xlabel('日对数收益率')
axes[1].set_ylabel('密度')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('沪深300日对数收益率（2005–2023）', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
```

```python
# ── 3. 用 QQ 图量化厚尾的程度 ─────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))
(osm, osr), (slope, intercept, r) = stats.probplot(ret, dist='norm')
ax.plot(osm, osr, '.', color='steelblue', alpha=0.3, markersize=3,
        label='数据分位数')
ax.plot(osm, slope * np.array(osm) + intercept, 
        color='tomato', lw=2, label='正态参考线')
ax.set_title('QQ 图：沪深300收益率 vs 正态分布')
ax.set_xlabel('正态理论分位数')
ax.set_ylabel('样本分位数')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# ── 4. 输出关键发现 ────────────────────────────────────────────
jb_stat, jb_p = stats.jarque_bera(ret)
print("\n=== 关键发现 ===")
print(f"偏度 = {ret.skew():.3f}  （正态应为 0，负值表示左偏）")
print(f"超额峰度 = {ret.kurtosis():.3f}  （正态应为 0，正值表示厚尾）")
print(f"Jarque-Bera 统计量 = {jb_stat:.1f}，p值 = {jb_p:.2e}")
print(f"  → p < 0.001，强烈拒绝正态假设")
```

**运行结果解读**：

从图形和统计量中，我们将观察到沪深300日收益率的三个典型特征，这在全球主要金融市场中普遍存在：

1. **左偏（负偏度）**：极端下跌比极端上涨更常发生——这在QQ图左尾的向上弯曲中清晰可见；
2. **厚尾（高峰度）**：正态分布严重低估了极端收益率的发生概率——QQ图两端均偏离参考线；
3. **Jarque-Bera检验**：以压倒性的统计显著性拒绝正态假设。

这一发现有直接的风险管理含义：**若用正态分布计算VaR，将系统性低估极端损失的概率。** 这正是2008年金融危机中许多风险模型集体失效的原因之一。

> **思考题**：经验分布（ECDF）本身是对总体分布的无偏估计吗？当样本量从500增加到4500时，ECDF与真实CDF之间的最大偏差如何变化？请用蒙特卡洛模拟验证 Glivenko-Cantelli 定理的收敛速度。

---

2.1 节到这里。下一节我们进入本章的核心框架——**分布的三种角色**，从数据分布出发，引出参数分布与误差项分布的区分，并以此揭示频率派与贝叶斯派的根本分歧。